# NB_LIBRO_12 · Figuras y experimentos del capítulo 10

**Libro:** *AgroVisión: del píxel a la decisión* · capítulo 10, «¿Se puede confiar en el modelo?»

Este notebook:

1. calcula los mapas Grad-CAM del clasificador de palma sobre las 70 imágenes de prueba y comprueba si señalan lo que el modelo usa (prueba de oclusión) y si se repiten entre semillas;
2. degrada las imágenes de prueba con ocho familias de perturbaciones y varios niveles, y evalúa los nueve clasificadores del proyecto en cada uno;
3. hace un experimento nuevo: entrena YOLO11m con copias desenfocadas de las imágenes de entrenamiento y mide si resiste mejor el desenfoque;
4. evalúa YOLO11s, YOLO26s y Faster R-CNN con un mismo protocolo y un mismo límite de detecciones, y mide su velocidad en igualdad de condiciones;
5. mide la velocidad de los clasificadores YOLO y de ResNet50 con la misma entrada.

**Usa (solo lectura):** `PALMA_AFRICANA_YOLO` (imágenes procesadas y `05_MODELOS`) y `AgroUTMACH_PIX` (validación del conjunto PIO y `03_MODELOS`). No modifica nada de los proyectos.

**Produce:** dos figuras, `valores_cap10.json` y varias tablas, en `AGROVISION_LIBRO`. Al final descarga un zip.

**Cómo ejecutarlo:** *Entorno de ejecución → Cambiar tipo de entorno → GPU* (L4 recomendada) y *Ejecutar todo*. Tarda de 25 a 40 minutos; la mitad es el entrenamiento del punto 3. Si Colab se desconecta, vuelva a ejecutar todo: los entrenamientos terminados se reutilizan. Suba al chat el zip que se descarga al final.

In [ ]:
%pip install -q ultralytics==8.4.172 scikit-learn

In [ ]:
import os, re, json, math, zipfile, time, shutil, gc, random, traceback, io
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))
LOCAL = Path(os.environ.get('AGROVISION_LOCAL', '/content'))
PALMA = BASE / 'PALMA_AFRICANA_YOLO'
DATOS = PALMA / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
DIR_MOD = PALMA / '05_MODELOS'
PIX = BASE / 'AgroUTMACH_PIX'
PIO = PIX / '01_DATASET_PIO' / 'PIO_TRAIN_VAL' / 'data'
MOD_Y11 = PIX / '03_MODELOS' / 'AgroUTMACH_PIX_YOLO11s_PIO_best.pt'
MOD_Y26 = PIX / '03_MODELOS' / 'AgroUTMACH_PIX_YOLO26s_PIO_best.pt'
MOD_FR = PIX / '03_MODELOS' / 'AgroUTMACH_PIX_FasterRCNN_ResNet50FPN_best.pt'
FR_ULTIMO = PIX / '05_RESULTADOS' / 'NB08_FASTER_RCNN_BENCHMARK' / 'checkpoint_last.pt'
DIR_FIG = BASE / 'AGROVISION_LIBRO' / '02_FIGURAS' / 'cap10'
DIR_VAL = BASE / 'AGROVISION_LIBRO' / '03_VALORES'
DIR_EXP = DIR_VAL / 'cap10_experimentos'
DIR_REC = DIR_EXP / 'recortes'
for d in (DIR_FIG, DIR_VAL, DIR_EXP, DIR_REC):
    d.mkdir(parents=True, exist_ok=True)
for p in (DATOS, DIR_MOD, PIO / 'images' / 'val', MOD_Y11, MOD_Y26, MOD_FR, FR_ULTIMO):
    print('OK   ' if p.exists() else 'FALTA', p)

ENTRENAR_CON_DESENFOQUE = True     # ponga False para omitir los seis entrenamientos (unos 15 minutos)
EVALUAR_DETECTORES = True          # ponga False para omitir la evaluación de los tres detectores
LADO, LOTE, EPOCAS, PACIENCIA = 384, 8, 60, 15          # configuración del proyecto para YOLO11m
SEMILLAS = [2026, 2027, 2028]
RAPIDO = os.environ.get('AGROVISION_RAPIDO') == '1'      # solo para pruebas del notebook
if RAPIDO:
    EPOCAS = 1

import torch, torchvision, ultralytics
from ultralytics import YOLO
from ultralytics.data.augment import classify_transforms
from sklearn.metrics import f1_score
HAY_GPU = torch.cuda.is_available()
DISP = 0 if HAY_GPU else 'cpu'
DEV = torch.device('cuda:0' if HAY_GPU else 'cpu')
GPU = torch.cuda.get_device_name(0) if HAY_GPU else 'sin GPU'
print('GPU:', GPU, '| PyTorch', torch.__version__, '| Torchvision', torchvision.__version__, '| Ultralytics', ultralytics.__version__)
assert HAY_GPU or not EN_COLAB, 'Active una GPU en Entorno de ejecución > Cambiar tipo de entorno.'

RAIZ = LOCAL / 'PALMA_cap10'
if RAIZ.exists():
    shutil.rmtree(RAIZ)
shutil.copytree(DATOS, RAIZ)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps
from skimage.color import rgb2lab, lab2rgb, rgb2hsv

ANCHO_CM = 16.0                      # ancho de caja de texto del libro
ANCHO_IN = ANCHO_CM / 2.54
DPI = 300

# Tinta y colores de apoyo. Los datos categóricos no se distinguen solo por color:
# las cinco clases se muestran en paneles separados (Figura 2.7).
TINTA = '#0b0b0b'
TINTA_2 = '#52514e'
GRIS_FONDO = '#c9c8c2'
ACENTO = '#2a78d6'
REJILLA = '#e4e3de'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA,
    'text.color': TINTA,
    'xtick.color': TINTA_2,
    'ytick.color': TINTA_2,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': DPI,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

CLASES = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']       # orden de presentación
NOMBRE_ES = {'unripe': 'Inmaduro', 'ripe': 'Maduro', 'overripe': 'Sobremaduro',
             'damaged': 'Dañado', 'empty': 'Vacío'}
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

VALORES = {}          # cifras que se copian al manuscrito
FIGURAS = []          # rutas de las figuras generadas


def guardar(fig, ruta):
    # Guarda la figura a 300 dpi y la registra.
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta))
    print('Guardada:', ruta.name)


def leer_rgb(ruta, lado_max=None):
    # Lee una imagen como arreglo RGB uint8, respetando la orientación EXIF.
    # Con lado_max, la reduce para que su lado mayor no supere ese valor.
    with Image.open(ruta) as im:
        if lado_max is not None and im.format == 'JPEG':
            im.draft('RGB', (lado_max * 2, lado_max * 2))     # decodificación rápida
        im = ImageOps.exif_transpose(im).convert('RGB')
        if lado_max is not None and max(im.size) > lado_max:
            im.thumbnail((lado_max, lado_max), Image.LANCZOS)
        return np.asarray(im)


def redimensionar(img, lado_mayor):
    # Redimensiona conservando la proporción hasta que el lado mayor mida lado_mayor.
    h, w = img.shape[:2]
    escala = lado_mayor / max(h, w)
    nuevo = (max(1, round(w * escala)), max(1, round(h * escala)))
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, nuevo, interpolation=interp)


def recorte_cuadrado(img, lado=512):
    # Recorte cuadrado centrado, redimensionado a lado × lado píxeles.
    h, w = img.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return cv2.resize(img[y0:y0 + m, x0:x0 + m], (lado, lado), interpolation=cv2.INTER_AREA)


def coma(x, dec=1):
    # Número con coma decimal, como se escribe en español.
    return f'{x:.{dec}f}'.replace('.', ',')


def eje_con_coma(eje, dec=2):
    eje.set_major_formatter(FuncFormatter(lambda v, _: coma(v, dec)))


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


print('Bibliotecas listas. matplotlib', matplotlib.__version__, '| OpenCV', cv2.__version__)

from PIL import ImageEnhance, ImageFilter

ERRORES = {}
CLASES_ORD = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']       # orden alfabético, el de Ultralytics
RUTAS = {p: sorted(q for q in (RAIZ / p).glob('*/*') if q.suffix.lower() in EXT_IMG) for p in ('train', 'val', 'test')}
print({p: len(v) for p, v in RUTAS.items()})
TF = classify_transforms(LADO)                  # redimensionar, recorte central, tensor y normalización
GEO = type(TF)(TF.transforms[:2])               # solo la parte geométrica: lo que el modelo ve
MODELOS = {}
for d in sorted(DIR_MOD.iterdir()):
    m = re.fullmatch(r'yolo11([nsm])_cls_seed(\d+)', d.name, flags=re.I)
    if m and (d / 'best.pt').exists():
        MODELOS[f'{m.group(1).lower()}_{m.group(2)}'] = d / 'best.pt'
print('Modelos del proyecto:', sorted(MODELOS))
assert 'm_2026' in MODELOS, 'Falta el modelo yolo11m_cls_seed2026 en 05_MODELOS.'


def seccion(nombre, fn):
    # Ejecuta una sección y, si falla, anota el error y deja que las demás continúen.
    t0 = time.time()
    try:
        fn()
        print(f'[{nombre}] terminada en {time.time() - t0:.0f} s')
    except Exception:
        ERRORES[nombre] = traceback.format_exc()[-1800:]
        print(f'[{nombre}] ERROR\n{ERRORES[nombre]}')
    plt.close('all')
    gc.collect()
    if HAY_GPU:
        torch.cuda.empty_cache()


def cargar(ruta):
    # Devuelve la red de un clasificador de Ultralytics, lista para inferencia, y el orden de sus clases.
    y = YOLO(str(ruta)); red = y.model.to(DEV).float().eval()
    nombres = [y.names[i] for i in range(len(y.names))]
    assert nombres == CLASES_ORD, nombres
    return red


def abrir(ruta):
    with Image.open(ruta) as im:
        return ImageOps.exif_transpose(im).convert('RGB')


NORMA = TF.transforms[-1]
MEDIA = torch.tensor(list(NORMA.mean), dtype=torch.float32).view(1, 3, 1, 1)
DESV = torch.tensor(list(NORMA.std), dtype=torch.float32).view(1, 3, 1, 1)


def a_u8(pils):
    # Recorte que ve el modelo, como tensor de enteros (N × 3 × 384 × 384): ocupa la cuarta parte que en coma flotante.
    return torch.from_numpy(np.stack([np.asarray(GEO(p)) for p in pils])).permute(0, 3, 1, 2).contiguous()


def normalizar(u8):
    return (u8.to(DEV).float().div(255) - MEDIA.to(DEV)) / DESV.to(DEV)


def salida(red, x):
    # En evaluación, el clasificador devuelve (probabilidades, puntuaciones); aquí se devuelven las puntuaciones.
    o = red(x)
    if isinstance(o, (tuple, list)):
        return o[1] if len(o) > 1 else o[0]
    return o


def probabilidades(red, x, lote=32):
    P = []
    with torch.no_grad():
        for k in range(0, len(x), lote):
            P.append(torch.softmax(salida(red, normalizar(x[k:k + lote])), 1).cpu().numpy())
    return np.concatenate(P)


def f1_macro(real, pred):
    return float(f1_score(real, pred, labels=CLASES_ORD, average='macro', zero_division=0))


PRUEBA = [abrir(p) for p in RUTAS['test']]
REAL = [p.parent.name for p in RUTAS['test']]
X0 = a_u8(PRUEBA)
VISTA = [x.permute(1, 2, 0).numpy() for x in X0]         # recorte de 384 × 384 que entra al modelo
assert float((normalizar(X0[:2]).cpu() - torch.stack([TF(p) for p in PRUEBA[:2]])).abs().max()) < 1e-4, 'El preprocesamiento no coincide con el de Ultralytics.'
for p, v in zip(RUTAS['test'], VISTA):
    Image.fromarray(v).save(DIR_REC / (p.stem + '.jpg'), quality=92)
pd.DataFrame({'imagen': [p.name for p in RUTAS['test']], 'real': REAL, 'ancho': [p.size[0] for p in PRUEBA], 'alto': [p.size[1] for p in PRUEBA]}).to_csv(DIR_EXP / 'prueba.csv', index=False)
print('Imágenes de prueba:', len(PRUEBA), '| tamaño de las procesadas:', sorted(set(p.size for p in PRUEBA))[:4])

## 1. Grad-CAM: dónde se apoya cada predicción

Grad-CAM pondera los mapas de la última capa convolucional con el gradiente de la puntuación de una clase. Aquí se programa directamente, sin bibliotecas adicionales, sobre la capa convolucional de la cabeza de clasificación.

In [ ]:
def gradcam(red, x, clase=None):
    # x: tensor de enteros 1 × 3 × H × W. Devuelve el mapa (h × w, normalizado a 1), las probabilidades y la clase usada.
    guardado = {}
    def gancho(modulo, entrada, sal):
        sal.retain_grad(); guardado['a'] = sal
    h = red.model[-1].conv.register_forward_hook(gancho)
    try:
        with torch.enable_grad():
            x = normalizar(x).requires_grad_(True)
            s = salida(red, x)
            c = int(s[0].argmax()) if clase is None else int(clase)
            red.zero_grad(set_to_none=True)
            s[0, c].backward()
        a = guardado['a']; g = a.grad
        pesos = g.mean(dim=(2, 3), keepdim=True)
        mapa = torch.relu((pesos * a).sum(1))[0].detach().cpu().numpy()
        p = torch.softmax(s.detach(), 1)[0].cpu().numpy()
    finally:
        h.remove()
    return mapa / max(float(mapa.max()), 1e-12), p, c


def superponer(vista, mapa, peso=0.55):
    m = cv2.resize(mapa.astype(np.float32), (vista.shape[1], vista.shape[0]), interpolation=cv2.INTER_CUBIC).clip(0, 1)
    color = (plt.get_cmap('inferno')(m)[..., :3] * 255).astype(np.float32)
    return (vista * (1 - peso * m[..., None]) + color * peso * m[..., None]).clip(0, 255).astype(np.uint8)


def ocluir(x, celdas, n):
    # Sustituye por el color medio de la imagen las celdas indicadas de una rejilla de n × n.
    x = x.clone(); media = x.float().mean(dim=(1, 2), keepdim=True).round().to(x.dtype); paso = x.shape[-1] // n
    for k in celdas:
        i, j = divmod(int(k), n)
        x[:, i * paso:(i + 1) * paso, j * paso:(j + 1) * paso] = media
    return x


def s_gradcam():
    red = cargar(MODELOS['m_2026'])
    # Comprobación: el preprocesamiento propio debe dar lo mismo que la llamada habitual de Ultralytics.
    P = probabilidades(red, X0)
    y = YOLO(str(MODELOS['m_2026'])); res = y.predict(source=[str(p) for p in RUTAS['test'][:8]], imgsz=LADO, device=DISP, verbose=False)
    Pu = np.array([r.probs.data.cpu().numpy() for r in res]); del y
    VALORES['base'] = {'exactitud': float(np.mean([CLASES_ORD[k] == r for k, r in zip(P.argmax(1), REAL)])),
                       'f1_macro': f1_macro(REAL, [CLASES_ORD[k] for k in P.argmax(1)]), 'dif_max_con_predict': float(np.abs(P[:8] - Pu).max())}
    print(VALORES['base'])
    mapas, mapas_real, filas = [], [], []
    rng = np.random.default_rng(2026)
    for i in range(len(PRUEBA)):
        m, p, c = gradcam(red, X0[i:i + 1]); n = m.shape[0]
        mr, _, _ = gradcam(red, X0[i:i + 1], CLASES_ORD.index(REAL[i]))
        mapas.append(m); mapas_real.append(mr)
        yy, xx = np.indices(m.shape); tot = m.sum() + 1e-12
        orden = np.argsort(-m.ravel()); k = round(0.25 * m.size)
        centro = np.zeros_like(m, bool); q = n // 4; centro[q:n - q, q:n - q] = True
        borde = np.ones_like(m, bool); borde[1:-1, 1:-1] = False
        acum = np.cumsum(np.sort(m.ravel())[::-1]) / tot
        # Prueba de oclusión: se tapa el 25 % de las celdas y se mira cuánto baja la probabilidad de la clase predicha.
        variantes = {'mas': orden[:k], 'menos': orden[-k:]}
        xs = [ocluir(X0[i], v, n) for v in variantes.values()] + [ocluir(X0[i], rng.choice(m.size, k, replace=False), n) for _ in range(5)]
        Po = probabilidades(red, torch.stack(xs))
        filas.append({'imagen': RUTAS['test'][i].name, 'real': REAL[i], 'pred': CLASES_ORD[c], 'conf': float(p[c]), 'celdas': int(m.size),
                      'cx': float((xx * m).sum() / tot / (n - 1)), 'cy': float((yy * m).sum() / tot / (n - 1)),
                      'masa_centro': float(m[centro].sum() / tot), 'area_centro': float(centro.mean()), 'masa_borde': float(m[borde].sum() / tot), 'area_borde': float(borde.mean()),
                      'celdas_mitad_masa': int(np.searchsorted(acum, 0.5) + 1), 'corr_pred_real': float(np.corrcoef(m.ravel(), mr.ravel())[0, 1]) if c != CLASES_ORD.index(REAL[i]) else 1.0,
                      'p_ocluir_mas': float(Po[0, c]), 'p_ocluir_menos': float(Po[1, c]), 'p_ocluir_azar': float(Po[2:, c].mean()),
                      'cambia_mas': bool(Po[0].argmax() != c), 'cambia_menos': bool(Po[1].argmax() != c), 'cambia_azar': float((Po[2:].argmax(1) != c).mean())})
    t = pd.DataFrame(filas); t.round(4).to_csv(DIR_EXP / 'gradcam_prueba.csv', index=False)
    np.savez_compressed(DIR_EXP / 'gradcam_m2026.npz', pred=np.array(mapas, np.float16), real=np.array(mapas_real, np.float16), probs=P.astype(np.float32))
    VALORES['gradcam'] = {'rejilla': list(mapas[0].shape), 'masa_centro': float(t.masa_centro.mean()), 'area_centro': float(t.area_centro.iloc[0]), 'masa_borde': float(t.masa_borde.mean()),
                          'area_borde': float(t.area_borde.iloc[0]), 'celdas_mitad_masa': float(t.celdas_mitad_masa.median()),
                          'por_clase': t.groupby('real')[['masa_centro', 'masa_borde', 'celdas_mitad_masa', 'conf']].mean().round(3).reset_index().to_dict('records'),
                          'oclusion': {'p_original': float(t.conf.mean()), 'p_mas': float(t.p_ocluir_mas.mean()), 'p_azar': float(t.p_ocluir_azar.mean()), 'p_menos': float(t.p_ocluir_menos.mean()),
                                       'cambia_mas': float(t.cambia_mas.mean()), 'cambia_azar': float(t.cambia_azar.mean()), 'cambia_menos': float(t.cambia_menos.mean())}}
    print({k: v for k, v in VALORES['gradcam'].items() if k != 'por_clase'})
    # Estabilidad: el mapa de la clase real con otros modelos del proyecto.
    otros = {}
    for clave in ('m_2027', 'm_2028', 's_2026', 'n_2026'):
        if clave in MODELOS:
            r2 = cargar(MODELOS[clave]); otros[clave] = np.array([gradcam(r2, X0[i:i + 1], CLASES_ORD.index(REAL[i]))[0] for i in range(len(PRUEBA))]); del r2
    ref = np.array(mapas_real); corr = {}
    for clave, M in otros.items():
        if M.shape == ref.shape:
            corr[clave] = float(np.nanmean([np.corrcoef(a.ravel(), b.ravel())[0, 1] for a, b in zip(ref, M)]))
    np.savez_compressed(DIR_EXP / 'gradcam_otros.npz', **{k: v.astype(np.float16) for k, v in otros.items()})
    perm = rng.permutation(len(ref))
    VALORES['gradcam']['correlacion_con_m_2026'] = corr
    VALORES['gradcam']['correlacion_entre_imagenes_distintas'] = float(np.nanmean([np.corrcoef(ref[i].ravel(), ref[j].ravel())[0, 1] for i, j in zip(range(len(ref)), perm) if i != j]))
    print('Correlación de mapas con m_2026:', corr, '| entre imágenes distintas:', round(VALORES['gradcam']['correlacion_entre_imagenes_distintas'], 3))
    # Fuera de dominio: un fotograma del galpón.
    pio = sorted(p for p in (PIO / 'images' / 'val').iterdir() if p.suffix.lower() in EXT_IMG)
    if pio:
        im = abrir(pio[0]); v = np.asarray(GEO(im)); m, p, c = gradcam(red, a_u8([im]))
        Image.fromarray(v).save(DIR_REC / 'fuera_de_dominio.jpg', quality=92); np.save(DIR_EXP / 'gradcam_fuera.npy', m.astype(np.float16))
        VALORES['gradcam']['fuera_de_dominio'] = {'archivo': pio[0].name, 'pred': CLASES_ORD[c], 'conf': float(p[c])}
    # Figura 1: un acierto por clase (el de confianza mediana).
    orden = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']; elegidos = []
    for c in orden:
        g = t[(t.real == c) & (t.pred == c)].sort_values('conf')
        if len(g):
            elegidos.append(int(g.index[len(g) // 2]))
    fig, ejes = plt.subplots(2, max(len(elegidos), 2), figsize=(ANCHO_IN, ANCHO_IN * 2 / max(len(elegidos), 2) + 0.35), squeeze=False)
    for j, i in enumerate(elegidos):
        ejes[0, j].imshow(VISTA[i]); ejes[1, j].imshow(superponer(VISTA[i], mapas[i]))
        ejes[0, j].set_title(f"{NOMBRE_ES[REAL[i]]} · {coma(t.conf[i], 2)}", pad=3)
    for ax in ejes.ravel():
        sin_ejes(ax)
    fig.subplots_adjust(wspace=0.04, hspace=0.04, left=0.005, right=0.995, top=0.93, bottom=0.005)
    guardar(fig, DIR_FIG / 'Fig_10_1_gradcam_aciertos.png'); plt.show()
    # Figura 2: los errores del modelo, con el mapa de la clase predicha y el de la clase real.
    err = t.index[t.real != t.pred].tolist()[:4]
    if err:
        fig, ejes = plt.subplots(3, max(len(err), 2), figsize=(ANCHO_IN, ANCHO_IN * 3 / max(len(err), 2) + 0.4), squeeze=False)
        for j, i in enumerate(err):
            ejes[0, j].imshow(VISTA[i]); ejes[1, j].imshow(superponer(VISTA[i], mapas[i])); ejes[2, j].imshow(superponer(VISTA[i], mapas_real[i]))
            ejes[0, j].set_title(f"Real: {NOMBRE_ES[REAL[i]].lower()}\nPredicho: {NOMBRE_ES[t.pred[i]].lower()} ({coma(t.conf[i], 2)})", pad=3, fontsize=7)
        for ax in ejes.ravel():
            sin_ejes(ax)
        fig.subplots_adjust(wspace=0.04, hspace=0.04, left=0.005, right=0.995, top=0.93, bottom=0.005)
        guardar(fig, DIR_FIG / 'Fig_10_2_gradcam_errores.png'); plt.show()
    VALORES['gradcam']['figura_aciertos'] = [RUTAS['test'][i].name for i in elegidos]; VALORES['gradcam']['figura_errores'] = [RUTAS['test'][i].name for i in err]

seccion('gradcam', s_gradcam)

## 2. Robustez: ocho familias de degradación y varios niveles

Cada degradación se aplica a la imagen procesada, antes del recorte, y se evalúan los nueve clasificadores del proyecto. La familia «resolución» reduce la imagen por un factor y la vuelve a ampliar, para imitar una fotografía de pocos píxeles.

In [ ]:
FAMILIAS = {'desenfoque': [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0], 'contraste': [0.85, 0.70, 0.55, 0.45, 0.30], 'ruido': [6, 12, 20, 28, 40],
            'resolucion': [2, 3, 4, 6, 8], 'oscurecer': [0.75, 0.55, 0.35], 'sobreexponer': [1.25, 1.45, 1.80], 'rotacion': [8, 18, 30], 'jpeg': [60, 30, 10]}
if RAPIDO:
    FAMILIAS = {k: v[1::2] for k, v in FAMILIAS.items()}
CONDICIONES = [('original', 0.0)] + [(f, float(n)) for f, niveles in FAMILIAS.items() for n in niveles]


def degradar(im, familia, nivel, semilla=0):
    # Las mismas operaciones que el notebook NB07 del proyecto, más la reducción de resolución.
    if familia == 'original':
        return im
    if familia in ('oscurecer', 'sobreexponer'):
        return ImageEnhance.Brightness(im).enhance(nivel)
    if familia == 'contraste':
        return ImageEnhance.Contrast(im).enhance(nivel)
    if familia == 'desenfoque':
        return im.filter(ImageFilter.GaussianBlur(radius=nivel))
    if familia == 'ruido':
        a = np.asarray(im, dtype=np.float32); rng = np.random.default_rng(semilla)
        return Image.fromarray(np.clip(a + rng.normal(0, nivel, a.shape), 0, 255).astype(np.uint8))
    if familia == 'rotacion':
        relleno = tuple(int(v) for v in np.median(np.asarray(im).reshape(-1, 3), axis=0))
        return im.rotate(nivel, resample=Image.Resampling.BICUBIC, expand=False, fillcolor=relleno)
    if familia == 'jpeg':
        b = io.BytesIO(); im.save(b, format='JPEG', quality=int(nivel)); b.seek(0)
        return Image.open(b).convert('RGB')
    if familia == 'resolucion':
        w, h = im.size
        return im.resize((max(8, round(w / nivel)), max(8, round(h / nivel))), Image.Resampling.LANCZOS).resize((w, h), Image.Resampling.BICUBIC)
    raise ValueError(familia)


def nitidez(vista):
    return float(cv2.Laplacian(cv2.cvtColor(vista, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var())


def tensores_degradados():
    # Tensores de las 70 imágenes en cada condición, y la nitidez del recorte que ve el modelo.
    T, nit = {}, []
    for k, (fam, niv) in enumerate(CONDICIONES):
        ims = [degradar(im, fam, niv, 2026 + 10000 * k + i) for i, im in enumerate(PRUEBA)]
        T[(fam, niv)] = a_u8(ims)
        for p, u in zip(RUTAS['test'], T[(fam, niv)]):
            nit.append({'familia': fam, 'nivel': niv, 'imagen': p.name, 'nitidez': nitidez(u.permute(1, 2, 0).numpy())})
    return T, pd.DataFrame(nit)


def evaluar_condiciones(red, T, **extra):
    filas = []
    for (fam, niv), x in T.items():
        P = probabilidades(red, x); k = P.argmax(1)
        d = pd.DataFrame({'familia': fam, 'nivel': niv, 'imagen': [p.name for p in RUTAS['test']], 'real': REAL, 'pred': [CLASES_ORD[i] for i in k], 'conf': P.max(1)})
        for j, c in enumerate(CLASES_ORD):
            d['p_' + c] = P[:, j]
        filas.append(d)
    return pd.concat(filas, ignore_index=True).assign(**extra)


def resumen_condiciones(t):
    g = t.groupby(['modelo', 'familia', 'nivel'], sort=False)
    return g.apply(lambda d: pd.Series({'exactitud': float((d.real == d.pred).mean()), 'f1_macro': f1_macro(d.real, d.pred), 'conf': float(d.conf.mean()),
                                        'conf_en_errores': float(d.conf[d.real != d.pred].mean()) if (d.real != d.pred).any() else np.nan,
                                        'errores_conf_090': int(((d.real != d.pred) & (d.conf > 0.9)).sum())})).reset_index()


TENS, NIT = None, None


def s_robustez():
    global TENS, NIT
    TENS, NIT = tensores_degradados(); NIT.round(2).to_csv(DIR_EXP / 'nitidez_por_condicion.csv', index=False)
    claves = sorted(MODELOS) if not RAPIDO else ['m_2026']
    trozos = []
    for clave in claves:
        red = cargar(MODELOS[clave]); trozos.append(evaluar_condiciones(red, TENS, modelo=clave)); del red
    t = pd.concat(trozos, ignore_index=True)
    t.round(5).to_csv(DIR_EXP / 'robustez_predicciones.csv', index=False)
    r = resumen_condiciones(t); r.round(4).to_csv(DIR_EXP / 'robustez_resumen.csv', index=False)
    media = r.groupby(['familia', 'nivel'], sort=False)[['exactitud', 'f1_macro', 'conf']].agg(['mean', 'min', 'max'])
    media.columns = ['_'.join(c) for c in media.columns]
    print(media.round(3).to_string())
    VALORES['robustez'] = {'modelos': claves, 'condiciones': len(CONDICIONES), 'media_por_condicion': media.reset_index().round(4).to_dict('records'),
                           'nitidez_mediana': NIT.groupby(['familia', 'nivel'], sort=False).nitidez.median().round(1).reset_index().to_dict('records')}

seccion('robustez', s_robustez)

## 3. Experimento: entrenar con desenfoque

El capítulo 3 dejó planteada una hipótesis: incluir desenfoque en el aumento de datos reduciría la vulnerabilidad más grave del modelo. Aquí se entrena YOLO11m con la configuración del proyecto y dos conjuntos de entrenamiento del mismo tamaño: uno en el que cada imagen aparece dos veces (control) y otro en el que la segunda copia está desenfocada. Validación y prueba no se tocan.

In [ ]:
def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def preparar_conjunto(nombre, desenfocar):
    destino = LOCAL / f'PALMA_{nombre}'
    if destino.exists():
        shutil.rmtree(destino)
    shutil.copytree(RAIZ, destino); rng = np.random.default_rng(2026); radios = []
    for p in sorted((destino / 'train').glob('*/*')):
        if p.suffix.lower() not in EXT_IMG:
            continue
        copia = p.with_name(p.stem + '_copia.jpg')
        if desenfocar:
            r = float(rng.uniform(0.5, 3.0)); radios.append(r)
            abrir(p).filter(ImageFilter.GaussianBlur(radius=r)).save(copia, quality=95)
        else:
            shutil.copy(p, copia)
    return destino, radios


def s_desenfoque():
    global TENS, NIT
    if not ENTRENAR_CON_DESENFOQUE:
        print('Entrenamientos omitidos.'); return
    if TENS is None:
        TENS, NIT = tensores_degradados()
    corridas = []
    for nombre, desenfocar in (('duplicado', False), ('desenfoque', True)):
        destino, radios = None, []
        for sem in SEMILLAS[:1] if RAPIDO else SEMILLAS:
            clave = f'{nombre}_{sem}'; rj, rc = DIR_EXP / f'entreno_{clave}.json', DIR_EXP / f'entreno_{clave}_pred.csv'
            if rj.exists() and rc.exists():
                print('REUTILIZADA:', clave); corridas.append(json.loads(rj.read_text(encoding='utf-8'))); continue
            if destino is None:
                destino, radios = preparar_conjunto(nombre, desenfocar)
            fijar_semilla(sem); t0 = time.time()
            YOLO('yolo11m-cls.pt' if not RAPIDO else 'yolo11n-cls.pt').train(data=str(destino), epochs=EPOCAS, imgsz=LADO if not RAPIDO else 64, batch=LOTE, patience=PACIENCIA, device=DISP, workers=2,
                                         seed=sem, deterministic=True, pretrained=True, optimizer='auto', amp=True, cache=True,
                                         project=str(LOCAL / 'cap10_runs'), name=clave, exist_ok=True, plots=False, verbose=False)
            minutos = (time.time() - t0) / 60
            mejor = LOCAL / 'cap10_runs' / clave / 'weights' / 'best.pt'
            assert mejor.exists(), f'No se generó {mejor}'
            red = cargar(mejor); t = evaluar_condiciones(red, TENS, modelo=clave)
            Pv = probabilidades(red, a_u8([abrir(p) for p in RUTAS['val']])); del red
            rcsv = LOCAL / 'cap10_runs' / clave / 'results.csv'
            r = {'clave': clave, 'conjunto': nombre, 'semilla': sem, 'minutos': round(minutos, 2), 'epocas': int(len(pd.read_csv(rcsv))) if rcsv.exists() else None,
                 'val_f1': f1_macro([p.parent.name for p in RUTAS['val']], [CLASES_ORD[k] for k in Pv.argmax(1)]),
                 'radio_medio': float(np.mean(radios)) if radios else None}
            t.round(5).to_csv(rc, index=False); rj.write_text(json.dumps(r, ensure_ascii=False, indent=1), encoding='utf-8')
            print(f"FINALIZADA: {clave} | {minutos:.1f} min | val F1 {r['val_f1']:.3f}")
            corridas.append(r)
    t = pd.concat([pd.read_csv(DIR_EXP / f"entreno_{r['clave']}_pred.csv") for r in corridas], ignore_index=True)
    r = resumen_condiciones(t); r['conjunto'] = r.modelo.str.split('_').str[0]
    r.round(4).to_csv(DIR_EXP / 'entreno_resumen.csv', index=False)
    tabla = r[r.familia.isin(['original', 'desenfoque'])].groupby(['conjunto', 'familia', 'nivel'], sort=False)[['exactitud', 'f1_macro']].mean().round(3)
    print(tabla.to_string())
    VALORES['entrenar_con_desenfoque'] = {'corridas': corridas, 'resumen': r.groupby(['conjunto', 'familia', 'nivel'], sort=False)[['exactitud', 'f1_macro', 'conf']].mean().round(4).reset_index().to_dict('records')}

seccion('desenfoque', s_desenfoque)

## 4. Tres detectores con un mismo protocolo

YOLO11s, YOLO26s y Faster R-CNN se evalúan sobre las 452 imágenes de validación con el mismo cálculo de la precisión media, el del capítulo 8, y con dos límites de detecciones por imagen: 1 000 y 100. La velocidad se mide imagen a imagen, descartando el arranque.

In [ ]:
def iou_matriz(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return (inter / (aa[:, None] + ab[None, :] - inter + 1e-12)).astype(np.float32)


def leer_cajas(ruta_txt, ancho, alto):
    filas = [l.split() for l in Path(ruta_txt).read_text(encoding='utf-8', errors='ignore').splitlines() if len(l.split()) == 5]
    if not filas:
        return np.zeros((0, 4), np.float32)
    a = np.array(filas, dtype=np.float32)[:, 1:]
    c = np.column_stack([(a[:, 0] - a[:, 2] / 2) * ancho, (a[:, 1] - a[:, 3] / 2) * alto, (a[:, 0] + a[:, 2] / 2) * ancho, (a[:, 1] + a[:, 3] / 2) * alto])
    return c[(c[:, 2] > c[:, 0]) & (c[:, 3] > c[:, 1])]


def emparejar(M, conf, umbral):
    par = np.full(len(conf), -1, np.int32); libre = np.ones(M.shape[1], bool)
    if M.shape[1] == 0:
        return par
    for k in np.argsort(-conf):
        fila = np.where(libre, M[k], -1.0); j = int(fila.argmax())
        if fila[j] >= umbral:
            par[k] = j; libre[j] = False
    return par


def precision_media(conf, tp, n_gt):
    orden = np.argsort(-conf); tp = tp[orden]
    acum_tp = np.cumsum(tp); acum_fp = np.cumsum(~tp)
    sens = acum_tp / max(n_gt, 1); prec = acum_tp / np.maximum(acum_tp + acum_fp, 1)
    env = np.maximum.accumulate(prec[::-1])[::-1]
    r = np.linspace(0, 1, 101); idx = np.searchsorted(sens, r, side='left')
    return float(np.where(idx < len(env), env[np.minimum(idx, len(env) - 1)], 0).mean())


UMBRALES_IOU = np.round(np.arange(0.5, 0.96, 0.05), 2)


def resumir(reg, tope):
    # reg: lista de imágenes con cajas, conf y gt. tope: máximo de detecciones por imagen (las de mayor confianza).
    cs, tps, n_gt, dif = [], {float(u): [] for u in UMBRALES_IOU}, 0, []
    for x in reg:
        o = np.argsort(-x['conf'])[:tope]; c = x['conf'][o]; M = iou_matriz(x['cajas'][o], x['gt'])
        cs.append(c); n_gt += len(x['gt']); dif.append(int((x['conf'] >= 0.25).sum()) - len(x['gt']))
        for u in UMBRALES_IOU:
            tps[float(u)].append(emparejar(M, c, u) >= 0)
    conf = np.concatenate(cs); ap = {u: precision_media(conf, np.concatenate(v), n_gt) for u, v in tps.items()}
    tp50 = np.concatenate(tps[0.5]); m = conf >= 0.25; dif = np.array(dif)
    return {'AP50': ap[0.5], 'AP75': ap[0.75], 'AP50_95': float(np.mean(list(ap.values()))), 'detecciones': int(len(conf)), 'anotaciones': int(n_gt),
            'precision_025': float(tp50[m].mean()) if m.any() else None, 'sensibilidad_025': float(tp50[m].sum() / max(n_gt, 1)),
            'conteo_eam': float(np.abs(dif).mean()), 'conteo_sesgo': float(dif.mean())}


def construir_faster():
    from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    m = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, min_size=640, max_size=960, box_detections_per_img=1000,
                                   rpn_pre_nms_top_n_train=3000, rpn_post_nms_top_n_train=2000, rpn_pre_nms_top_n_test=3000, rpn_post_nms_top_n_test=1500)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 2)
    return m


def sincronizar():
    if HAY_GPU:
        torch.cuda.synchronize()


def s_detectores():
    if not EVALUAR_DETECTORES:
        print('Evaluación de detectores omitida.'); return
    rutas = sorted(p for p in (PIO / 'images' / 'val').iterdir() if p.suffix.lower() in EXT_IMG)
    if RAPIDO:
        rutas = rutas[:6]
    gts, tams = [], []
    for p in rutas:
        with Image.open(p) as im:
            w, h = im.size
        tams.append((w, h)); gts.append(leer_cajas(PIO / 'labels' / 'val' / (p.stem + '.txt'), w, h))
    VALORES['detectores'] = {}; n_tiempo = min(100, len(rutas))

    def anotar(nombre, reg, extra):
        r = {'limite_1000': resumir(reg, 1000), 'limite_100': resumir(reg, 100), **extra}
        VALORES['detectores'][nombre] = r
        print(f"{nombre:22s} AP50 {r['limite_1000']['AP50']:.3f} | AP50-95 {r['limite_1000']['AP50_95']:.3f} | con 100 detecciones {r['limite_100']['AP50_95']:.3f} | "
              f"conteo EAM {r['limite_1000']['conteo_eam']:.1f} sesgo {r['limite_1000']['conteo_sesgo']:+.1f} | {extra}")
        if nombre.startswith('faster'):          # las de YOLO11s ya se guardaron en el capítulo 8
            img = np.concatenate([np.full(len(x['conf']), i, np.int16) for i, x in enumerate(reg)]); cj = np.concatenate([x['cajas'] for x in reg]); cf = np.concatenate([x['conf'] for x in reg])
            q = cf >= 0.05
            np.savez_compressed(DIR_EXP / f'detecciones_{nombre}.npz', img=img[q], caja=np.round(cj[q] * 10).astype(np.int16), conf=np.round(cf[q] * 255).astype(np.uint8))

    for nombre, ruta in (('yolo11s', MOD_Y11), ('yolo26s', MOD_Y26)):
        try:
            det = YOLO(str(ruta)); reg = []; ms = {'preprocess': [], 'inference': [], 'postprocess': []}; pared = []
            for i, p in enumerate(rutas):
                t0 = time.perf_counter()
                r = det.predict(source=str(p), imgsz=960, conf=0.001, iou=0.7, max_det=1000, device=DISP, verbose=False)[0]
                sincronizar(); dt = (time.perf_counter() - t0) * 1000
                reg.append({'cajas': r.boxes.xyxy.cpu().numpy().astype(np.float32), 'conf': r.boxes.conf.cpu().numpy().astype(np.float32), 'gt': gts[i]})
                if 10 <= i < 10 + n_tiempo:
                    pared.append(dt)
                    for k in ms:
                        ms[k].append(float(r.speed.get(k, np.nan)))
            anotar(nombre, reg, {'ms_pared': float(np.mean(pared)) if pared else None, **{'ms_' + k: float(np.mean(v)) if v else None for k, v in ms.items()},
                                 'parametros_M': float(sum(q.numel() for q in det.model.parameters()) / 1e6), 'archivo_mb': round(ruta.stat().st_size / 1e6, 1)})
            del det, reg
        except Exception:
            VALORES['detectores'][nombre] = {'error': traceback.format_exc()[-1200:]}; print(nombre, 'ERROR\n', VALORES['detectores'][nombre]['error'])
    variantes = [('faster_rcnn_mejor', MOD_FR, None), ('faster_rcnn_ultima', FR_ULTIMO, 'model_state')]
    for nombre, ruta, campo in variantes:
        try:
            if not ruta.exists():
                print('No está', ruta); continue
            estado = torch.load(ruta, map_location='cpu', weights_only=False); info = {}
            if campo:
                info = {'epoca': int(estado.get('epoch', -1)) + 1, 'historial': estado.get('history')}; estado = estado[campo]
            mod = construir_faster(); mod.load_state_dict(estado); del estado
            mod.roi_heads.score_thresh = 0.001; mod = mod.to(DEV).eval(); reg = []; pared = []
            with torch.inference_mode():
                for i, p in enumerate(rutas):
                    x = torchvision.transforms.functional.to_tensor(abrir(p)).to(DEV); sincronizar(); t0 = time.perf_counter()
                    o = mod([x])[0]; sincronizar(); dt = (time.perf_counter() - t0) * 1000
                    reg.append({'cajas': o['boxes'].cpu().numpy().astype(np.float32), 'conf': o['scores'].cpu().numpy().astype(np.float32), 'gt': gts[i]})
                    if 10 <= i < 10 + n_tiempo:
                        pared.append(dt)
            anotar(nombre, reg, {'ms_pared': float(np.mean(pared)) if pared else None, 'parametros_M': float(sum(q.numel() for q in mod.parameters()) / 1e6),
                                 'archivo_mb': round(ruta.stat().st_size / 1e6, 1), **info})
            del mod, reg
        except Exception:
            VALORES['detectores'][nombre] = {'error': traceback.format_exc()[-1200:]}; print(nombre, 'ERROR\n', VALORES['detectores'][nombre]['error'])
    VALORES['detectores']['imagenes'] = len(rutas)

seccion('detectores', s_detectores)

## 5. Velocidad de los clasificadores con la misma entrada

Se mide solo el paso de la imagen por la red, con tensores ya cargados en la GPU, con dos tamaños de imagen y dos tamaños de lote.

In [ ]:
def s_velocidad():
    from torchvision.models import resnet50
    redes = {}
    for clave in ('n_2026', 's_2026', 'm_2026'):
        if clave in MODELOS:
            redes['YOLO11' + clave[0]] = cargar(MODELOS[clave])
    for d in sorted(DIR_MOD.iterdir()):
        m = re.fullmatch(r'yolo26([nsm])_cls_seed2026', d.name, flags=re.I)
        if m and (d / 'best.pt').exists():
            try:
                redes['YOLO26' + m.group(1).lower()] = cargar(d / 'best.pt')
            except Exception as e:
                print('No se pudo cargar', d.name, e)
    r50 = resnet50(weights=None); r50.fc = torch.nn.Linear(r50.fc.in_features, 5); redes['ResNet50'] = r50.to(DEV).eval()
    filas = []; rep = 5 if RAPIDO else 40
    for nombre, red in redes.items():
        n_par = sum(q.numel() for q in red.parameters()) / 1e6
        for lado in (224, 384):
            for lote in (1, 32):
                if RAPIDO and lote == 32:
                    continue
                x = torch.rand(lote, 3, lado, lado, device=DEV)
                with torch.no_grad():
                    for _ in range(5):
                        red(x)
                    sincronizar(); t0 = time.perf_counter()
                    for _ in range(rep):
                        red(x)
                    sincronizar()
                filas.append({'modelo': nombre, 'parametros_M': round(n_par, 2), 'lado': lado, 'lote': lote, 'ms_por_imagen': (time.perf_counter() - t0) * 1000 / rep / lote})
    t = pd.DataFrame(filas); t.round(4).to_csv(DIR_EXP / 'velocidad_clasificadores.csv', index=False)
    print(t.pivot_table(index=['modelo', 'parametros_M'], columns=['lado', 'lote'], values='ms_por_imagen').round(2).to_string())
    VALORES['velocidad_clasificadores'] = t.to_dict('records')

seccion('velocidad', s_velocidad)

## 6. Guardar y descargar

In [ ]:
VALORES['errores'] = ERRORES
VALORES['entorno'] = {'gpu': GPU, 'torch': torch.__version__, 'torchvision': torchvision.__version__, 'ultralytics': ultralytics.__version__}
ruta_json = DIR_VAL / 'valores_cap10.json'
ruta_json.write_text(json.dumps(VALORES, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
ruta_zip = LOCAL / 'AgroVision_cap10.zip'
with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DIR_FIG.glob('Fig_10_*.png')):
        z.write(f, 'cap10/' + f.name)
    z.write(ruta_json, ruta_json.name)
    for f in sorted(DIR_EXP.rglob('*')):
        if f.is_file():
            z.write(f, 'cap10_experimentos/' + str(f.relative_to(DIR_EXP)))
print('Secciones con error:', list(ERRORES) or 'ninguna')
print('Figuras:', len(FIGURAS), '· zip de', round(ruta_zip.stat().st_size / 1e6, 1), 'MB')
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_zip))